# Academic Conference Keyword Filter
This script scans an Excel dataset and separates rows into two groups based on whether they contain academic event keywords (e.g., 'Symposium', 'Tagung', 'Workshop').

### Features:
- **Multilingual Support:** Includes keywords in German, English, French, and Italian.
- **Unicode Normalization:** Correctly matches words regardless of accents or casing.

In [ ]:
import pandas as pd
import re
import unicodedata
import os

# === 1. CONFIGURATION ===
input_path = "../data/input_entities.xlsx"
output_clean = "../outputs/data_without_conferences.xlsx"
output_conferences = "../outputs/detected_conferences_only.xlsx"

# Comprehensive list of academic event markers
ACADEMIC_KEYWORDS = [
    "conference", "meeting", "symposium", "colloquium", "colloque", "colloquy",
    "congress", "kongress", "congres", "congresso", "congrès", "tagung", "workshop",
    "seminar", "séminaire", "consultation", "assembly", "summit", "session",
    "sitzung", "journées", "field conference", "round table", "rundtisch",
    "table ronde", "studientagung", "arbeitstagung", "fachtagung",
    "ringvorlesung", "advanced seminar", "studienkreis",
    "wissenschaftliches gespräch", "wissenschaftliche konferenz",
    "wissenschaftliches kolloquium", "forschungskonferenz",
    "forschungskolloquium", "cours général public", "jahresversammlung",
    "convegno", "tagung", "vorlesung", "ausstellung", "course", "congregatio",
    "plenum", "versammlung", "tagung", "summer", "sommer", "settimana", "konferenz",
    "journée", "incontro", "synode", "symposion", "messe", "gespräch", "kolloquium"
]

# Function to strip accents and lowercase text
def normalize(text):
    if not isinstance(text, str):
        return ""
    nfkd = unicodedata.normalize('NFKD', text)
    return ''.join([c for c in nfkd if not unicodedata.combining(c)]).lower()

# Compile the search pattern
normalized_keywords = [normalize(k) for k in ACADEMIC_KEYWORDS]
pattern = re.compile('|'.join(map(re.escape, normalized_keywords)), re.IGNORECASE)

# === 2. DATA PROCESSING ===
print(f"📂 Loading file: {input_path}...")
df = pd.read_excel(input_path)

def row_matches(row):
    for val in row:
        if pattern.search(normalize(val)):
            return True
    return False

print("🔍 Scanning rows for keywords...")
mask = df.apply(row_matches, axis=1)

# Split the data
konferenz_rows = df[mask]
filtered_df = df[~mask]

# === 3. SAVE OUTPUTS ===
filtered_df.to_excel(output_clean, index=False)
konferenz_rows.to_excel(output_conferences, index=False)

print("✅ Done!")
print(f"– Clean Data: {output_clean} ({len(filtered_df)} rows)")
print(f"– Detected Conferences: {output_conferences} ({len(konferenz_rows)} rows)")